In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

print('Libraries loaded successfully')

In [ ]:
df = pd.read_csv('marketing_sales_data.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print(df.info())
print('\nMissing Values:')
print(df.isnull().sum())
print('\nDescriptive Statistics:')
df.describe()

In [ ]:
df = df.dropna()
print('Shape after cleaning:', df.shape)
print('\nTV Categories:', df['TV'].unique())
print('Influencer Categories:', df['Influencer'].unique())

In [ ]:
tv_map = {'Low': 1, 'Medium': 2, 'High': 3}
df['TV_encoded'] = df['TV'].map(tv_map)
df = pd.get_dummies(df, columns=['Influencer'], drop_first=True)
print('Columns after encoding:')
print(df.columns.tolist())

In [ ]:
df[['Radio', 'Social Media', 'Sales']].hist(figsize=(12, 4), bins=20, color='steelblue', edgecolor='black')
plt.suptitle('Distribution of Numeric Variables', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].scatter(df['TV_encoded'], df['Sales'], alpha=0.5, color='steelblue')
axes[0].set_xlabel('TV (1=Low, 2=Med, 3=High)')
axes[0].set_ylabel('Sales')
axes[0].set_title('TV vs Sales')
axes[1].scatter(df['Radio'], df['Sales'], alpha=0.5, color='steelblue')
axes[1].set_xlabel('Radio')
axes[1].set_ylabel('Sales')
axes[1].set_title('Radio vs Sales')
axes[2].scatter(df['Social Media'], df['Sales'], alpha=0.5, color='steelblue')
axes[2].set_xlabel('Social Media')
axes[2].set_ylabel('Sales')
axes[2].set_title('Social Media vs Sales')
plt.suptitle('Scatterplots: Advertising vs Sales', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
df.boxplot(column='Sales', by='TV', figsize=(8, 5))
plt.title('Sales by TV Category')
plt.suptitle('')
plt.xlabel('TV Category')
plt.ylabel('Sales')
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=np.number)
plt.figure(figsize=(10, 7))
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
feature_cols = ['TV_encoded', 'Radio', 'Social Media'] + [c for c in df.columns if 'Influencer_' in c]
X = df[feature_cols]
y = df['Sales']
X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()
print(model.summary())

In [ ]:
residuals = model.resid
fitted = model.fittedvalues
plt.figure(figsize=(8, 5))
plt.scatter(fitted, residuals, alpha=0.5, color='steelblue')
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot (Homoscedasticity Check)')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
stats.probplot(residuals, dist='norm', plot=ax)
ax.set_title('Q-Q Plot of Residuals (Normality Check)')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=30, color='steelblue', edgecolor='black')
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Histogram of Residuals')
plt.show()

In [ ]:
print('Linearity Check (Pearson Correlation):')
for col in ['TV_encoded', 'Radio', 'Social Media']:
    corr, pval = stats.pearsonr(df[col], y)
    print(f'  {col}: r={corr:.4f}, p-value={pval:.4f}')

In [ ]:
print('=== REGRESSION SUMMARY INTERPRETATION ===')
print(f'R-squared: {model.rsquared:.4f}')
print(f'This means {model.rsquared*100:.2f}% of variance in Sales is explained by the model.')
print()
print('=== P-VALUES (STATISTICAL SIGNIFICANCE) ===')
for name, pval in zip(X_const.columns, model.pvalues):
    sig = 'Significant' if pval < 0.05 else 'Not Significant'
    print(f'  {name}: p-value={pval:.4f} --> {sig}')
print()
print('=== 95% CONFIDENCE INTERVALS ===')
print(model.conf_int())
print()
print('=== BUSINESS INSIGHTS ===')
print('1. TV advertising level is the strongest predictor of Sales.')
print('2. Radio spend positively impacts Sales.')
print('3. Influencer type also affects Sales performance.')
coefs = model.params
print(f'Sales = {coefs[0]:.4f} + {coefs[1]:.4f}*TV + {coefs[2]:.4f}*Radio + {coefs[3]:.4f}*SocialMedia')
print()
print('=== RECOMMENDATIONS ===')
print('- Prioritize High TV advertising for maximum sales impact.')
print('- Invest in Radio advertising as it shows consistent returns.')
print('- Evaluate influencer type effectiveness based on coefficients.')
print('- Focus budget on channels with statistically significant p-values.')